# 02 - Preprocessing
Subset, regrid, normalize CMEMS and AIS data to the WPS ConvLSTM model domain.

**Inputs (from Notebook 01):** `physics_raw_region.nc`, `bgc_raw_region.nc`, `ais_raw_region.parquet`

**Outputs:** `preprocessed_features.nc` (7 channels, 41x25 grid, monthly)

In [1]:
# CELL: imports, drive mount, and shared pipeline CONFIG
from google.colab import drive
import xarray as xr
import pandas as pd
import numpy as np
from scipy.interpolate import interp1d

drive.mount('/content/drive')

CONFIG = {
    # ------------------------------------------------------------------
    # Spatial
    # ------------------------------------------------------------------
    # Broad regional bbox: matches the cache written by Notebook 01.
    'bbox_regional': {
        'lat_min': 0,   'lat_max': 30,
        'lon_min': 110, 'lon_max': 140,
    },
    # WPS model target bbox: all ConvLSTM inputs are subset to this region.
    # Narrow domain -> higher positive-pixel density -> better F1 score.
    'bbox_model': {
        'lat_min': 10,  'lat_max': 20,
        'lon_min': 114, 'lon_max': 120,
    },

    # ------------------------------------------------------------------
    # Temporal
    # ------------------------------------------------------------------
    # Full date range for raw CMEMS/AIS load (Notebook 01).
    'date_full': {'start': '2014-01-01', 'end': '2024-12-31'},
    # ConvLSTM training window: starts 2019 for denser AIS coverage.
    'date_model': {'start': '2019-01-01', 'end': '2024-12-31'},

    # ------------------------------------------------------------------
    # Paths
    # ------------------------------------------------------------------
    'data_dir': '/content/drive/MyDrive/fishing_project/',
    'files': {
        # Source CMEMS NetCDF files (placed in data_dir by the user)
        'physics_w_nc':  'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779636039565.nc',
        'physics_ht_nc': 'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779635380319.nc',
        'bgc_src_nc':    'cmems_mod_glo_bgc_my_0.25deg_P1M-m_1779635372583.nc',
        # Notebook 01 outputs -> Notebook 02 inputs
        'physics_nc':    'physics_raw_region.nc',
        'bgc_nc':        'bgc_raw_region.nc',
        'ais_parquet':   'ais_raw_region.parquet',
        'ais_csv_gz':    'ais_raw_region.csv.gz',
        # Notebook 02 outputs -> Notebook 03 inputs
        'ais_gridded_nc':  'ais_fishing_effort_gridded.nc',
        'preprocessed_nc': 'preprocessed_features.nc',
    },

    # ------------------------------------------------------------------
    # AIS
    # ------------------------------------------------------------------
    'ais_use_cols': ['date', 'cell_ll_lat', 'cell_ll_lon', 'fishing_hours'],

    # ------------------------------------------------------------------
    # Depth selection for CMEMS variables
    # ------------------------------------------------------------------
    'physics_surface_depth': 0.49,   # metres, nearest-neighbour selection
    'bgc_depth_range': (0.51, 5.14), # metres, averaged over this range

    # ------------------------------------------------------------------
    # Preprocessing
    # ------------------------------------------------------------------
    'norm_method': 'minmax',
    'resample_freq': '1ME',
}

DATA_DIR = CONFIG['data_dir']
bbox_m   = CONFIG['bbox_model']
dates_m  = CONFIG['date_model']
print(f'DATA_DIR   : {DATA_DIR}')
print(f'Model bbox : {bbox_m}')
print(f'Model dates: {dates_m}')

Mounted at /content/drive
DATA_DIR   : /content/drive/MyDrive/fishing_project/
Model bbox : {'lat_min': 10, 'lat_max': 20, 'lon_min': 114, 'lon_max': 120}
Model dates: {'start': '2019-01-01', 'end': '2024-12-31'}


## STEP 1: Load and Subset Datasets

In [2]:
# CELL: load regional datasets from Notebook 01, subset to WPS model bbox and dates
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
f  = CONFIG['files']

# Load the pre-merged regional datasets saved by Notebook 01
physics_ds = xr.open_dataset(DATA_DIR + f['physics_nc'])
bgc_ds     = xr.open_dataset(DATA_DIR + f['bgc_nc'])

phys_depths = physics_ds.depth.values[:5]
bgc_depths  = bgc_ds.depth.values[:5]
print(f'Available physics depths : {phys_depths}')
print(f'Available BGC depths     : {bgc_depths}')

# Physics: select nearest surface depth, then subset to WPS model bbox and dates
surf_depth = CONFIG['physics_surface_depth']
physics_subset = physics_ds.sel(
    depth=surf_depth, method='nearest'
).sel(
    latitude=slice(bm['lat_min'], bm['lat_max']),
    longitude=slice(bm['lon_min'], bm['lon_max']),
    time=slice(dm['start'], dm['end'])
)

# BGC: average over near-surface depths, then subset to WPS model bbox and dates
d_lo, d_hi = CONFIG['bgc_depth_range']
bgc_subset = bgc_ds.sel(
    depth=slice(d_lo, d_hi),
    latitude=slice(bm['lat_min'], bm['lat_max']),
    longitude=slice(bm['lon_min'], bm['lon_max']),
    time=slice(dm['start'], dm['end'])
).mean(dim='depth')

phys_dims = dict(physics_subset.sizes)
bgc_dims  = dict(bgc_subset.sizes)
print(f'Physics subset (0.083 deg) : {phys_dims}')
print(f'BGC subset    (0.25 deg)  : {bgc_dims}')

Available physics depths : [0.494025 1.541375 2.645669 3.819495 5.078224]
Available BGC depths     : [0.50576   1.5558553 2.6676817 3.8562799 5.1403613]
Physics subset (0.083 deg) : {'time': 72, 'latitude': 121, 'longitude': 72}
BGC subset    (0.25 deg)  : {'time': 72, 'latitude': 41, 'longitude': 25}


/tmp/ipykernel_14219/1310457616.py:34: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  phys_dims = dict(physics_subset.dims)
/tmp/ipykernel_14219/1310457616.py:35: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  bgc_dims  = dict(bgc_subset.dims)


## STEP 2: Monthly Aggregation

In [10]:
# CELL: resample to monthly means
# CMEMS data is already monthly, but resampling ensures consistent time-stamp
# labels and suppresses the FutureWarning from the deprecated '1M' string.
# CONFIG['resample_freq'] = '1ME' is the modern replacement.
freq = CONFIG['resample_freq']

physics_monthly = physics_subset.resample(time=freq).mean()
bgc_monthly     = bgc_subset.resample(time=freq).mean()

print(f'Monthly physics : {dict(physics_monthly.sizes)}')
print(f'Monthly BGC     : {dict(bgc_monthly.sizes)}')

Monthly physics : {'time': 72, 'latitude': 121, 'longitude': 72}
Monthly BGC     : {'time': 72, 'latitude': 41, 'longitude': 25}


## STEP 3: Define Target Grid and Regrid

In [4]:
# CELL: regrid physics (0.083 deg) onto the BGC native grid (0.25 deg)
# Using BGC's native coordinates as the canonical target grid so all
# 7 ConvLSTM input channels share the same spatial resolution and extent.
target_lats = bgc_monthly.latitude.values
target_lons = bgc_monthly.longitude.values

n_lat     = len(target_lats)
n_lon     = len(target_lons)
lat_range = (target_lats.min(), target_lats.max())
lon_range = (target_lons.min(), target_lons.max())
print(f'Target grid : {n_lat} x {n_lon} at 0.25 deg resolution')
print(f'  Lat range : {lat_range[0]:.2f} to {lat_range[1]:.2f}')
print(f'  Lon range : {lon_range[0]:.2f} to {lon_range[1]:.2f}')

# Bilinear interpolation of physics to match BGC grid
physics_regrid = physics_monthly.interp(
    latitude=target_lats,
    longitude=target_lons,
    method='linear'
)

print(f'Physics regridded : {dict(physics_regrid.sizes)}')

Target grid : 41 x 25 at 0.25 deg resolution
  Lat range : 10.00 to 20.00
  Lon range : 114.00 to 120.00
Physics regridded : {'time': 72, 'latitude': 41, 'longitude': 25}


/tmp/ipykernel_14219/3624965094.py:22: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  print(f'Physics regridded : {dict(physics_regrid.dims)}')


## STEP 4: Verify Alignment

In [5]:
# CELL: assert grid and time alignment before downstream processing
assert np.allclose(physics_regrid.latitude,  bgc_monthly.latitude),  'Latitude mismatch!'
assert np.allclose(physics_regrid.longitude, bgc_monthly.longitude), 'Longitude mismatch!'
assert len(physics_regrid.time) == len(bgc_monthly.time),            'Time-step count mismatch!'

n_months = len(bgc_monthly.time)
n_pixels = n_lat * n_lon
print(f'All grids aligned -- {n_months} months at 0.25 deg resolution')
print(f'  Grid size : {n_lat} lat x {n_lon} lon = {n_pixels:,} pixels/month')

All grids aligned -- 72 months at 0.25 deg resolution
  Grid size : 41 lat x 25 lon = 1,025 pixels/month


## STEP 5: Gap Filling

In [6]:
# CELL: fill NaN gaps with linear temporal interpolation
def fill_gaps(data):
    '''Fill NaN gaps with linear interpolation along the time axis.
    extrapolate handles edge months with no valid neighbours.'''
    return data.interpolate_na(dim='time', method='linear', fill_value='extrapolate')

physics_filled = fill_gaps(physics_regrid)
bgc_filled     = fill_gaps(bgc_monthly)

# Report any remaining NaNs per variable after filling
for ds_name, ds in [('physics', physics_filled), ('bgc', bgc_filled)]:
    for var in ds.data_vars:
        n_nan  = int(np.isnan(ds[var].values).sum())
        status = 'OK' if n_nan == 0 else f'WARNING {n_nan} NaNs remain'
        print(f'  {ds_name}.{var}: {status}')

print('Gap filling complete.')

  physics.uo: WARNING 6768 NaNs remain
  physics.vo: WARNING 6768 NaNs remain
  physics.zos: WARNING 6768 NaNs remain
  physics.thetao: WARNING 6768 NaNs remain
  bgc.chl: WARNING 1152 NaNs remain
  bgc.nppv: WARNING 1152 NaNs remain
Gap filling complete.


## STEP 6: Extract and Normalize Variables

In [7]:
# CELL: extract all 6 oceanographic channels and normalize
# Normalization method is controlled by CONFIG['norm_method'].
# 'minmax' -> scales each variable to [0, 1] globally over the full time series.
# 'zscore' -> centres at mean=0, std=1 over the full time series.
def normalize(data, method=None):
    '''Normalize a DataArray using the method specified in CONFIG.'''
    if method is None:
        method = CONFIG['norm_method']
    if method == 'minmax':
        mn, mx = float(data.min()), float(data.max())
        return (data - mn) / (mx - mn) if mx > mn else data * 0
    elif method == 'zscore':
        return (data - data.mean()) / data.std()
    raise ValueError(f'Unknown normalization method: {method}')

# Extract variables from BGC dataset (biogeochemical)
chl  = normalize(bgc_filled['chl'])   # Chlorophyll-a
nppv = normalize(bgc_filled['nppv'])  # Net primary production

# Extract variables from Physics dataset
ssh = normalize(physics_filled['zos'])     # Sea surface height
sst = normalize(physics_filled['thetao'])  # Sea surface temperature
uo  = normalize(physics_filled['uo'])      # Eastward sea water velocity
vo  = normalize(physics_filled['vo'])      # Northward sea water velocity

method_used = CONFIG['norm_method']
print(f'Normalization complete (method={method_used})')
for var_name, arr in [('Chl', chl), ('NPPV', nppv), ('SSH', ssh),
                       ('SST', sst), ('UO',  uo),   ('VO',  vo)]:
    arr_min = float(arr.min())
    arr_max = float(arr.max())
    print(f'  {var_name:4s}: shape={arr.shape}  min={arr_min:.3f}  max={arr_max:.3f}')

Normalization complete (method=minmax)
  Chl : shape=(72, 41, 25)  min=0.000  max=1.000
  NPPV: shape=(72, 41, 25)  min=0.000  max=1.000
  SSH : shape=(72, 41, 25)  min=0.000  max=1.000
  SST : shape=(72, 41, 25)  min=0.000  max=1.000
  UO  : shape=(72, 41, 25)  min=0.000  max=1.000
  VO  : shape=(72, 41, 25)  min=0.000  max=1.000


## STEP 7: Process AIS Data

In [8]:
# CELL: load pre-filtered AIS Parquet from Notebook 01, subset to WPS model bbox
# and date window, aggregate to 0.25 deg monthly grid, log-normalise, cache as NetCDF.
# NOTE: load_ais_filtered() is NOT redefined here -- Notebook 01 already filtered
# and saved ais_raw_region.parquet. We simply load and narrow that cache.
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
f  = CONFIG['files']

# --- 7a: Load from Notebook 01 output (Parquet preferred, CSV.gz fallback) ---
ais_parquet_name = f['ais_parquet']
ais_csv_gz_name  = f['ais_csv_gz']
print('Loading pre-filtered AIS data from Notebook 01...')
try:
    ais_df = pd.read_parquet(DATA_DIR + ais_parquet_name)
    print(f'  Loaded Parquet : {ais_parquet_name}')
except Exception:
    ais_df = pd.read_csv(DATA_DIR + ais_csv_gz_name)
    print(f'  Loaded CSV.gz fallback : {ais_csv_gz_name}')

# --- 7b: Narrow the regional AIS cache to the WPS model bbox ---
mask = (
    (ais_df['cell_ll_lat'] >= bm['lat_min']) & (ais_df['cell_ll_lat'] <  bm['lat_max']) &
    (ais_df['cell_ll_lon'] >= bm['lon_min']) & (ais_df['cell_ll_lon'] <  bm['lon_max'])
)
ais_df = ais_df[mask].copy()
ais_df['date']       = pd.to_datetime(ais_df['date'])
ais_df['year_month'] = ais_df['date'].dt.to_period('M')

# Further filter to the model date window (2019-2024)
date_mask = (
    (ais_df['date'] >= dm['start']) &
    (ais_df['date'] <= dm['end'])
)
ais_df = ais_df[date_mask]

n_records    = len(ais_df)
ais_min_date = ais_df['date'].min().date()
ais_max_date = ais_df['date'].max().date()
n_months_ais = ais_df['year_month'].nunique()
print(f'AIS records in WPS model bbox : {n_records:,}')
print(f'Date range                    : {ais_min_date} to {ais_max_date}')
print(f'Unique months                 : {n_months_ais}')


# --- 7c: Aggregate 0.1 deg GFW rows -> 0.25 deg monthly grid ---
def aggregate_ais_to_grid(ais_df, target_lats, target_lons):
    '''Bin AIS fishing_hours into the 0.25 deg target grid per month.
    GFW lower-left cell corners are shifted +0.05 to get cell centres.'''
    lat_c = ais_df['cell_ll_lat'].values + 0.05
    lon_c = ais_df['cell_ll_lon'].values + 0.05

    lat_idx = (np.searchsorted(target_lats, lat_c) - 1).clip(0, len(target_lats) - 1)
    lon_idx = (np.searchsorted(target_lons, lon_c) - 1).clip(0, len(target_lons) - 1)

    df = ais_df.copy()
    df['lat_idx'] = lat_idx
    df['lon_idx'] = lon_idx

    result = {}
    for period, grp in df.groupby('year_month'):
        grid = np.zeros((len(target_lats), len(target_lons)), dtype=np.float32)
        np.add.at(grid,
                  (grp['lat_idx'].values, grp['lon_idx'].values),
                  grp['fishing_hours'].values)
        result[str(period)] = grid
    return result


def ais_to_xarray(ais_grids, cmems_times, target_lats, target_lons):
    '''Align monthly AIS grids to the CMEMS time axis; missing months filled with 0.'''
    data = np.zeros((len(cmems_times), len(target_lats), len(target_lons)), dtype=np.float32)
    for i, t in enumerate(cmems_times):
        key = str(pd.Timestamp(t).to_period('M'))
        if key in ais_grids:
            data[i] = ais_grids[key]
    return xr.DataArray(
        data,
        dims=['time', 'latitude', 'longitude'],
        coords={'time': cmems_times, 'latitude': target_lats, 'longitude': target_lons},
        name='fishing_hours'
    )


def normalize_ais(da):
    '''Log1p then min-max normalization.
    Log1p compresses the heavy-tailed fishing effort distribution before scaling.
    Keeps zero-effort pixels at 0 after normalization.'''
    log_da = np.log1p(da)
    mn, mx = float(log_da.min()), float(log_da.max())
    return (log_da - mn) / (mx - mn) if mx > mn else log_da * 0


ais_grids      = aggregate_ais_to_grid(ais_df, target_lats, target_lons)
ais_da         = ais_to_xarray(ais_grids, bgc_monthly.time.values, target_lats, target_lons)
ais_normalized = normalize_ais(ais_da)

n_months_agg = len(ais_grids)
ais_da_sizes = dict(ais_da.sizes)
ais_norm_min = float(ais_normalized.min())
ais_norm_max = float(ais_normalized.max())
print(f'Months aggregated : {n_months_agg}')
print(f'AIS DataArray     : {ais_da_sizes}')
print(f'AIS normalized    : min={ais_norm_min:.3f}  max={ais_norm_max:.3f}')

# Cache gridded (un-normalised) AIS to Drive -- skips re-aggregation on restart
ais_gridded_nc_name = f['ais_gridded_nc']
ais_da.to_netcdf(DATA_DIR + ais_gridded_nc_name)
print(f'Saved AIS grid : {ais_gridded_nc_name}')

Loading pre-filtered AIS data from Notebook 01...
  Loaded Parquet : ais_raw_region.parquet
AIS records in WPS model bbox : 141,381
Date range                    : 2019-01-01 to 2024-12-01
Unique months                 : 72
Months aggregated : 72
AIS DataArray     : {'time': 72, 'latitude': 41, 'longitude': 25}
AIS normalized    : min=0.000  max=1.000
Saved AIS grid : ais_fishing_effort_gridded.nc


## STEP 8: Save Preprocessed Data

In [9]:
# CELL: combine all 7 channels into one xarray Dataset and save to NetCDF
# This file is the sole input to 03_model_training.ipynb.
preprocessed = xr.Dataset({
    'chl':            chl,             # Chlorophyll-a              (BGC)
    'nppv':           nppv,            # Net primary production      (BGC)
    'ssh':            ssh,             # Sea surface height          (Physics)
    'sst':            sst,             # Sea surface temperature     (Physics)
    'uo':             uo,              # Eastward sea water velocity (Physics)
    'vo':             vo,              # Northward sea water velocity (Physics)
    'fishing_effort': ais_normalized,  # Log-normalised AIS fishing hours (AIS)
})

preprocessed_nc_name = CONFIG['files']['preprocessed_nc']
out_path = DATA_DIR + preprocessed_nc_name
preprocessed.to_netcdf(out_path)

final_dims = dict(preprocessed.sizes)
final_vars = list(preprocessed.data_vars)
print(f'Preprocessed data saved : {preprocessed_nc_name}')
print(f'  Dimensions : {final_sizes}')
print(f'  Variables  : {final_vars}')
print('Notebook 02 complete. Run 03_model_training.ipynb next.')

Preprocessed data saved : preprocessed_features.nc
  Dimensions : {'latitude': 41, 'longitude': 25, 'time': 72}
  Variables  : ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']
Notebook 02 complete. Run 03_model_training.ipynb next.


/tmp/ipykernel_14219/1750788356.py:17: FutureWarning: The return type of `Dataset.dims` will be changed to return a set of dimension names in future, in order to be more consistent with `DataArray.dims`. To access a mapping from dimension names to lengths, please use `Dataset.sizes`.
  final_dims = dict(preprocessed.dims)
